# Vortex Fitting and Detection Notebook

This notebook runs the vortex fitting script on GLORYS data using the 'thesis' kernel.

In [7]:
import sys
import os
sys.path.append('SOFTX-D-20-00015-master')

import argparse
from vortexfitting import fitting
from vortexfitting import schemes
from vortexfitting import detection
from vortexfitting import output
from vortexfitting import classes
import xarray as xr

In [8]:
import matplotlib
matplotlib.use('Agg')  # For headless plotting

In [9]:
# # Set Parameters — Delta criterion
# input_filename = 'SOFTX-D-20-00015-master/data/GPGP_oct2020_22-27N_145-140W2.nc'
# output_directory = 'results'
# scheme = 22
# detection_method = 'delta'
# detection_threshold = 0.0
# box_size = 8
# flip_axis = False
# mean_filename = '/'
# plot_method = 'fit'
# xy_location = [0, 0]
# first = 0
# last = 0
# step = 1
# rmax = 0
# file_type = 'dns'
# correlation_threshold = 0.6
# output_format = 'png'

In [10]:
# # Set Parameters — Q criterion
# input_filename = 'SOFTX-D-20-00015-master/data/GPGP_aug2020_22-27N_145-140W.nc'
# output_directory = 'results'
# scheme = 22
# detection_method = 'Q'
# detection_threshold = 0.00000000000001
# box_size = 12
# flip_axis = False
# mean_filename = '/'
# plot_method = 'fit'
# xy_location = [0, 0]
# first = 0
# last = 0
# step = 1
# rmax = 0
# file_type = 'dns'
# correlation_threshold = 0.5
# output_format = 'png'

In [11]:
# Set Parameters swirling method
input_filename = './data/GPGP_aug2020_22-23N_140-139W.nc'
output_directory = 'results'
scheme = 4
detection_method = 'swirling'
detection_threshold = 0.0
box_size = 7
flip_axis = False
mean_filename = '/'
plot_method = 'fit'
xy_location = [0, 0]
first = 0
last = 0
step = 1
rmax = 0 # 8000
file_type = 'dns'
correlation_threshold = 0.5   
output_format = 'png'

In [12]:
ds_test = xr.open_dataset(input_filename)
print(ds_test)

<xarray.Dataset> Size: 6kB
Dimensions:     (time: 1, depth: 1, latitude: 21, longitude: 23)
Coordinates:
  * time        (time) int64 8B 0
  * depth       (depth) float64 8B 0.0
  * latitude    (latitude) float64 168B 0.0 9.266e+03 ... 1.761e+05 1.853e+05
  * longitude   (longitude) float64 184B 0.0 8.531e+03 ... 1.791e+05 1.877e+05
Data variables:
    velocity_x  (time, depth, latitude, longitude) float32 2kB ...
    velocity_y  (time, depth, latitude, longitude) float32 2kB ...
    velocity_z  (time, depth, latitude, longitude) float32 2kB ...
Attributes:
    Conventions:       CF-1.11
    title:             daily mean fields from Global Ocean Physics Analysis a...
    institution:       MERCATOR OCEAN
    source:            MERCATOR GLORYS12V1
    history:           2023/06/01 16:20:05 MERCATOR OCEAN Netcdf creation
    references:        http://www.mercator-ocean.fr
    comment:           CMEMS product
    subset:source:     ARCO data downloaded from the Marine Data Store using ...

In [13]:
# Load Data
if last < first:
    last = first

for time_step in range(first, last + 1, step):
    if not os.path.exists(input_filename.format(time_step)):
        print('The input file does not exist. Exiting.')
        sys.exit()

    print('\nOpening file: ', input_filename.format(time_step), ', file type: ', file_type)
    if mean_filename != '/':
        print('Opening mean field: ', mean_filename)

    vfield = classes.VelocityField(input_filename, time_step, mean_filename, file_type)


Opening file:  ./data/GPGP_aug2020_22-23N_140-139W.nc , file type:  dns


In [14]:
# Compute Derivatives
if scheme == 4:
    vfield.derivative = schemes.fourth_order_diff(vfield)
elif scheme == 2:
    vfield.derivative = schemes.second_order_diff(vfield)
elif scheme == 22:
    vfield.derivative = schemes.least_square_diff(vfield)
else:
    print('No scheme', scheme, 'found. Exiting!')
    sys.exit()

Difference scheme: Fourth Order Scheme


In [15]:
# Compute Vorticity
vorticity = vfield.derivative['dvdx'] - vfield.derivative['dudy']

In [16]:
# Detect Vortices
detection_field = []
if detection_method == 'Q':
    detection_field = detection.calc_q_criterion(vfield)
elif detection_method == 'swirling':
    detection_field = detection.calc_swirling(vfield)
elif detection_method == 'delta':
    detection_field = detection.calc_delta_criterion(vfield)

if vfield.normalization_flag:
    print('Normalization for ', vfield.normalization_direction, ' direction')
    detection_field = fitting.normalize(detection_field, vfield.normalization_direction)

Detection method: 2D swirling strength
Max value of swirling:  0.0


In [17]:
# Find Peaks
print('Threshold=', detection_threshold, ', box size=', box_size)
peaks = fitting.find_peaks(detection_field, detection_threshold, box_size)
print('Vortices found: ', len(peaks[0]))

Threshold= 0.0 , box size= 7
Vortices found:  5


In [18]:
# Determine Rotation Direction
vortices_counterclockwise, vortices_clockwise = fitting.direction_rotation(vorticity, peaks)

In [19]:
# Fit Vortices
vortices = list()
if (plot_method == 'fit') and (xy_location == [0, 0]):
    vortices = fitting.get_vortices(vfield, peaks, vorticity, rmax, correlation_threshold)
    print('---- Accepted vortices ----')
    print(len(vortices))
else:
    print('No fitting')

0 Processing detected swirling at (x, y) 2 2
1 Processing detected swirling at (x, y) 17 8
Accepted! Correlation = 0.81 (vortex # 0)
2 Processing detected swirling at (x, y) 4 12
Accepted! Correlation = 0.63 (vortex # 1)
3 Processing detected swirling at (x, y) 14 16
4 Processing detected swirling at (x, y) 10 18
Accepted! Correlation = 0.52 (vortex # 2)
---- Accepted vortices ----
3


In [20]:
# Plot Results
if xy_location != [0, 0]:
    x_location = int(xy_location[0])
    y_location = int(xy_location[1])
    detection_field_window = detection_field[y_location - 10:y_location + 10, x_location - 10:x_location + 10]
    x_index, y_index, u_data, v_data = fitting.window(vfield, x_location, y_location, 10)
    fitting.plot_quiver(x_index, y_index, u_data, v_data, detection_field_window)
if plot_method == 'detect':
    fitting.plot_detect(vortices_counterclockwise, vortices_clockwise, detection_field, flip_axis)
if plot_method == 'fields':
    fitting.plot_fields(vfield, vorticity)
if plot_method == 'fit':
    os.makedirs(output_directory, exist_ok=True)
    fitting.plot_accepted(vfield, vortices, detection_field, output_directory, time_step, output_format)
    fitting.plot_vortex(vfield, vortices, output_directory, time_step, output_format)
    output.write(vortices, output_directory, time_step)

r: 40308.021 gamma: 57891.67 xc: 122024.17 yc: 74130.09 correlation: 0.81 utheta: 0.14
r: 15874.358 gamma: -8631.27 xc: 35048.70 yc: 111194.93 correlation: 0.63 utheta: -0.05
r: 26798.241 gamma: 19741.57 xc: 86300.95 yc: 166792.39 correlation: 0.52 utheta: 0.07
